# NL2SHACL-Bench: End-to-End Tutorial

This notebook walks through the complete NL2SHACL-Bench pipeline, from raw SHACL shapes to a benchmark evaluation. It uses the example data in `examples/my-dataset/` and produces a dataset in the same format as `examples/example-nl2shacl-dataset/`.

The pipeline has two main stages:
- **Stage 1: Dataset Construction** — extract shape fragments, generate natural language descriptions, and produce a structured dataset.
- **Stage 2: Translation and Evaluation** — run a translation system on the dataset and evaluate the results.

---
**To use your own data**, replace the contents of `examples/my-dataset/shacl/` and `examples/my-dataset/ontology/` with your own files before running Stage 1.

## Prerequisites

In [ ]:
# Install required packages
!pip install rdflib pyshacl requests rdf-graph-gen google-genai

In [ ]:
import os

# Set paths
FRAMEWORK_ROOT = "."  # adjust if running from a different directory
EXAMPLE_DIR    = os.path.join(FRAMEWORK_ROOT, "examples", "my-dataset")
DATASET_DIR    = os.path.join(FRAMEWORK_ROOT, "examples", "example-nl2shacl-dataset")

print(f"Framework root : {os.path.abspath(FRAMEWORK_ROOT)}")
print(f"Example dir    : {os.path.abspath(EXAMPLE_DIR)}")
print(f"Dataset output : {os.path.abspath(DATASET_DIR)}")

---
## Stage 1: Dataset Construction

### Step 1: Extract shape fragments

Extract top-level node shapes from the raw SHACL file. All `.ttl` files in `shacl/` are merged into a single graph before extraction, so cross-file references are resolved correctly.

In [ ]:
!python Dataset-Construction/Data-Preprocessor/convert_shacl.py \
    --input-dir examples/my-dataset/shacl \
    --output-file examples/my-dataset/output_data.jsonl

In [ ]:
# Preview the first extracted record
import json

with open("examples/my-dataset/output_data.jsonl", encoding="utf-8") as f:
    first = json.loads(f.readline())

print(f"ID    : {first['id']}")
print(f"NL    : {first['nl']}")
print(f"SHACL :\n{first['shacl'][:300]}...")

### Step 2: Quality check and prefix extraction

Check structural completeness of extracted fragments and extract all prefix declarations. Review the log before proceeding — remove any `FAIL` entries from `output_data.jsonl` if needed.

In [ ]:
!python Dataset-Construction/Data-Preprocessor/shacl_quality_check.py \
    examples/my-dataset/output_data.jsonl

In [ ]:
# Print the last 20 lines of the check log
with open("examples/my-dataset/output_data_check_log.txt", encoding="utf-8") as f:
    lines = f.readlines()
print("".join(lines[-20:]))

### Step 3: Augment with ontology metadata

Look up ontology term metadata for each URI referenced in the shapes.

In [ ]:
!python Dataset-Construction/Data-Preprocessor/ontology_augment.py \
    examples/my-dataset/output_data.jsonl \
    --prefixes examples/my-dataset/output_data_prefixes.json \
    --ontology-dir examples/my-dataset/ontology

In [ ]:
# Preview the ontology snippet added to the first record
with open("examples/my-dataset/output_data_augmented.jsonl", encoding="utf-8") as f:
    first_aug = json.loads(f.readline())

print(f"ID: {first_aug['id']}")
print(f"Ontology snippet ({len(first_aug['ontology_snippet'])} terms):")
for uri, meta in list(first_aug['ontology_snippet'].items())[:2]:
    print(f"  {uri}")
    print(f"    label: {meta.get('label', '—')}")

### Step 4: Generate description prompts

Generate LLM prompts for each shape fragment. Use `--subset` for a built-in domain role, or `--role` to provide a custom role sentence.

In [ ]:
!python Dataset-Construction/Description-Generator/get_nl_prompt.py \
    --input examples/my-dataset/output_data_augmented.jsonl \
    --subset snik

### Step 5: Call Gemini API to generate descriptions

> **Note:** This step requires a Gemini API key. Set the `GEMINI_API_KEY` environment variable before running.
> If you do not have an API key, skip this cell — the pre-generated file `examples/my-dataset/output_data_augmented_nl_prompts.jsonl` is already available and Step 6 will use it directly.

In [ ]:
# Skip this cell if you do not have a Gemini API key.
# Set your API key first:
# os.environ["GEMINI_API_KEY"] = "your_api_key_here"

!python Dataset-Construction/Description-Generator/run_gemini.py \
    --input examples/my-dataset/output_data_augmented_nl_prompts.jsonl

### Step 6: Human review

> **Note:** The Description Reviewer is a GUI tool and must be run outside this notebook:
> ```bash
> python Dataset-Construction/Description-Reviewer/UI_annotator.py
> ```
> For this tutorial, we use the pre-reviewed file already available in `examples/my-dataset/`.

In [ ]:
# Preview a reviewed description
reviewed_path = "examples/my-dataset/output_data_description_reviewed.jsonl"

with open(reviewed_path, encoding="utf-8") as f:
    first_reviewed = json.loads(f.readline())

print(f"ID          : {first_reviewed['id']}")
print(f"Description : {first_reviewed['description']}")

### Step 7: Convert to dataset format

Combine the augmented JSONL and the reviewed descriptions into the final structured dataset format.

In [ ]:
!python Dataset-Construction/convert_dataset.py \
    examples/my-dataset/output_data_augmented.jsonl \
    --descriptions examples/my-dataset/output_data_description_reviewed.jsonl \
    --out-dir examples/example-nl2shacl-dataset \
    --prefix snik

In [ ]:
# Verify the output structure
for fname in sorted(os.listdir("examples/example-nl2shacl-dataset")):
    print(fname)

shacl_files = os.listdir("examples/example-nl2shacl-dataset/shacl")
print(f"\nshacl/ contains {len(shacl_files)} .ttl files")

---
## Stage 2: Translation and Evaluation

This stage evaluates a translation system on the dataset produced in Stage 1. We use the minimal rule-based translator as a reference example. To evaluate your own system, follow `NL2SHACL-Translator/TRANSLATOR_SPEC.md`.

### Step 8: Run the translator

The rule-based translator reads the dataset and produces translated SHACL shapes. It requires no API access.

In [ ]:
!python NL2SHACL-Translator/translator_example_rule_based.py \
    --subset snik-dataset \
    --dataset-dir examples/example-nl2shacl-dataset

In [ ]:
# Preview the translator output
with open("your_outputs.jsonl", encoding="utf-8") as f:
    first_output = json.loads(f.readline())

print(f"ID           : {first_output['id']}")
print(f"Output SHACL :\n{first_output.get('output_shacl', '(null)')[:300]}")

### Step 9: Attach reference SHACL

Attach the reference shapes from the dataset to each translated record.

In [ ]:
os.makedirs("processed-output", exist_ok=True)

!python NL2SHACL-Translator/attach_reference.py \
    --input your_outputs.jsonl \
    --dataset-dir examples/example-nl2shacl-dataset \
    --subset snik-dataset \
    --output processed-output/snik-dataset_rule_based_processed.jsonl

### Step 10: Run evaluation

In [ ]:
!python Shapes-Evaluation/run_evaluation.py \
    --input processed-output/snik-dataset_rule_based_processed.jsonl

### Step 11: Compute metrics

In [ ]:
!python Shapes-Evaluation/compute_metrics.py \
    --input evaluation-output/

In [ ]:
# Display the metrics summary
import csv

with open("metrics-output/metrics_summary.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

for row in rows:
    print(row)

---
## Summary

| Step | Script | Input | Output |
|------|--------|-------|--------|
| 1 | `convert_shacl.py` | `shacl/` | `output_data.jsonl` |
| 2 | `shacl_quality_check.py` | `output_data.jsonl` | `output_data_check_log.txt`, `output_data_prefixes.json` |
| 3 | `ontology_augment.py` | `output_data.jsonl` | `output_data_augmented.jsonl` |
| 4 | `get_nl_prompt.py` | `output_data_augmented.jsonl` | `output_data_augmented_nl_prompts.jsonl` |
| 5 | `run_gemini.py` | `output_data_augmented_nl_prompts.jsonl` | `..._generated_description.jsonl` |
| 6 | `UI_annotator.py` | generated descriptions | `output_data_description_reviewed.jsonl` |
| 7 | `convert_dataset.py` | augmented + reviewed | `snik-descriptions.jsonl`, `snik-ontology_snippets.jsonl`, `shacl/` |
| 8 | `translator_example_rule_based.py` | dataset | `your_outputs.jsonl` |
| 9 | `attach_reference.py` | `your_outputs.jsonl` | `processed-output/xxx_processed.jsonl` |
| 10 | `run_evaluation.py` | `processed-output/` | `evaluation-output/xxx_eval.jsonl` |
| 11 | `compute_metrics.py` | `evaluation-output/` | `metrics-output/metrics_summary.csv` |